# CrossQSD

In [1]:
import sys
sys.path.append("../")

In [2]:
from flow.solve_mix import *
from flow.interface import *
from utils.handy_states import *

In [3]:
# Define input states
state_info = coh_asymm_small(num_qubits=3)
num_qubits = state_info["num_qubits"]
num_states = state_info["num_states"]
state_vec = state_info["state_vec"]
dense_mat = state_info["dense_mat"]

In [4]:
qsd_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
)

In [13]:
qsd_problem.set_states(
    state_type="statevector",
    states=state_vec,
    overwrite=True,
)

cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": 1e-8}
ideal_result = apply_Eldar(qsd_problem, cvxpy_settings=cvxpy_settings)
np.format_float_positional(ideal_result["p_succ"], 6)

/home/koova22/miniconda3/envs/q1/lib/python3.12/site-packages/mosek/__init__.py:18617: UserWarning: Argument sub in putvarboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putvarboundlist: Incorrect array format causing data to be copied");
/home/koova22/miniconda3/envs/q1/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/koova22/miniconda3/envs/q1/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");
sol[1] is zero or negative (5.965104728453773e-10 <= 1e-4), skip its operator


'0.234321'

In [6]:
ideal_result["sol"]

array([3.5148e-01, 5.9651e-10, 3.5148e-01])

In [7]:
# Fix apply_Eldar to place 0 matrix instead of skipping the operator
ideal_povm = [
    ideal_result["povm"][0],
    np.zeros(shape=(8, 8)),
    ideal_result["povm"][1],
    np.identity(8) - ideal_result["povm"][0] - ideal_result["povm"][1],
]

print(verify_povm_matrix(ideal_povm))

ideal_prob_mat = calculate_prob_matrix_simple(
    prior_probs=[1 / num_states] * num_states,
    povm=ideal_povm,
    states=dense_mat,
)
ideal_p_succ = 0
for i in range(num_states):
    ideal_p_succ += ideal_prob_mat[i][i]
print(f"p_succ = {ideal_p_succ:.4f}")
for line in ideal_prob_mat:
    print(line)
print()

True
p_succ = 0.2343
[0.11716065984558524, 0.0, 2.8527929749198703e-18, 0.21617267348774807]
[1.7593896487353747e-18, 0.0, 8.922124091953508e-19, 0.3333333333333333]
[1.7702847014022026e-18, 0.0, 0.11716065984558484, 0.2161726734877484]



In [8]:
# Calculate the noise effect on the POVMs
disturbance_states = [
    DensityMatrix(
        ProblemSpec.depolarizing_noise_channel(num_qubits=num_qubits)
    )
    for _ in range(num_states)
]

In [15]:
params = [0.1 ** (6 - 0.25 * i) for i in range(25)]
def test_crossQSD(noise_param, param):
    print(f"tol = {np.format_float_scientific(param, 5)}")
    noisy_dense_mat = [
        (1 - noise_param) * dense_mat[_]
        + noise_param * disturbance_states[_].data
        for _ in range(num_states)
    ]
    qsd_problem.set_states(
        state_type="densitymatrix",
        states=noisy_dense_mat,
        overwrite=True,
    )

    cvxpy_crossqsd_problem = apply_crossQSD(
        qsd_problem,
        alpha=[param] * num_states,
        beta=[param] * num_states,
        cvxpy_settings={"solver": cp.MOSEK, "verbose": False, "eps": 1e-8},
    )

    vars = cvxpy_crossqsd_problem.variables()
    povm = [var.value for var in vars]
    prob_mat = calculate_prob_matrix_simple(
        prior_probs=[1 / num_states] * num_states,
        povm=povm,
        states=dense_mat,
    )
    p_succ = 0
    p_inc = 0
    for i in range(num_states):
        p_succ += prob_mat[i][i]
        p_inc += prob_mat[i][num_states]
    p_err = 1 - p_succ - p_inc
    print(f"p_succ = {p_succ:.5f}")
    print(f"p_err = {p_err:.5f}")
    print(f"p_err / p_succ = {np.format_float_scientific(p_err / p_succ, 5)}")
    # for line in prob_mat:
    #     print(line)
    print()
    return povm, p_succ

In [23]:
povm, theo_p_succ = test_crossQSD(params[16], params[16])

tol = 1.00000e-02
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11439e-03



In [24]:
disturbance_states_dense_mat = [
    m.data for m in disturbance_states
]
noise_prob_mat = calculate_prob_matrix_simple(
    prior_probs=[1 / num_states] * num_states,
    povm=povm,
    states=disturbance_states_dense_mat,
)
for item in noise_prob_mat:
    print(item)
noise_p = 0
for i in range(num_states):
    noise_p += noise_prob_mat[i][i]
print(noise_p)

[0.022564369286176045, 0.017383028524430702, 0.022564369287768844, 0.2708215660594533]
[0.022564369286176045, 0.017383028524430702, 0.022564369287768844, 0.2708215660594533]
[0.022564369286176045, 0.017383028524430702, 0.022564369287768844, 0.2708215660594533]
0.06251176709837558


In [25]:
# Test different depolarizing noise 0-25
print(np.format_float_scientific(params[16], 5))
for noise_param in params:
    if noise_param == params[8]:
        print(f"---")
    noisy_dense_mat = [
        (1 - noise_param) * dense_mat[_]
        + noise_param * disturbance_states[_].data
        for _ in range(num_states)
    ]
    prob_mat = calculate_prob_matrix_simple(
        prior_probs=[1 / num_states] * num_states,
        povm=povm,
        states=noisy_dense_mat,
    )
    p_succ = 0
    p_inc = 0
    for i in range(num_states):
        p_succ += prob_mat[i][i]
        p_inc += prob_mat[i][num_states]
    p_err = 1 - p_succ - p_inc
    print(rf"\lambda = {np.format_float_scientific(noise_param, 5)}")
    print(f"p_succ = {p_succ:.5f}")
    print(f"p_err = {p_err:.5f}")
    print(f"p_err / p_succ = {np.format_float_scientific(p_err / p_succ, 5)}")
    print("worst-case p_succ", np.format_float_positional((1 - noise_param) * theo_p_succ + noise_param * noise_p, 5))
    # for line in prob_mat:
    #     print(line)
    print()

1.00000e-02
\lambda = 1.00000e-06
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11489e-03
worst-case p_succ 0.25197

\lambda = 1.77828e-06
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11527e-03
worst-case p_succ 0.25197

\lambda = 3.16228e-06
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11596e-03
worst-case p_succ 0.25197

\lambda = 5.62341e-06
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11718e-03
worst-case p_succ 0.25197

\lambda = 1.00000e-05
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.11934e-03
worst-case p_succ 0.25197

\lambda = 1.77828e-05
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.1232e-03
worst-case p_succ 0.25197

\lambda = 3.16228e-05
p_succ = 0.25197
p_err = 0.00129
p_err / p_succ = 5.13005e-03
worst-case p_succ 0.25197

\lambda = 5.62341e-05
p_succ = 0.25196
p_err = 0.00130
p_err / p_succ = 5.14223e-03
worst-case p_succ 0.25196

---
\lambda = 1.00000e-04
p_succ = 0.25195
p_err = 0.00130
p_err / p_succ = 5.16389e-03
worst-case p_